# Extract period strings

Every string the transformers turn into a Period, from the EBSCO, Axiell and FOLIO adapter stores,
written to `data/periods.jsonl` as one row per distinct `(store, tag, path, text)` with its
occurrence count. `source` is `axiell` for Axiell strings and `marc` for the rest, the one distinction the
parsing rules depend on.

In [1]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [2]:
import json
from collections import Counter
from pathlib import Path

from adapters.extractors.ebsco.helpers import build_adapter_table as build_ebsco_table
from adapters.extractors.oai_pmh.axiell.runtime import AXIELL_CONFIG
from adapters.extractors.oai_pmh.folio.runtime import FOLIO_CONFIG
from adapters.utils.adapter_store import AdapterStore
from utils.marc import parse_single_marc_record

OUT = Path("data/periods.jsonl")

TABLES = {
    "ebsco": build_ebsco_table(use_rest_api_table=True),
    "axiell": AXIELL_CONFIG.build_adapter_table(use_rest_api_table=True),
    "folio": FOLIO_CONFIG.build_adapter_table(use_rest_api_table=True),
}

PERIOD_SUBFIELDS = {
    "648": ("subject", "ay"),
    "650": ("subject", "y"),
    "651": ("subject", "y"),
    "655": ("genre", "y"),
    "260": ("production", "cg"),
    "264": ("production", "c"),
}

In [3]:
def period_strings(record):
    for field in record.get_fields(*PERIOD_SUBFIELDS):
        path, codes = PERIOD_SUBFIELDS[field.tag]
        for value in field.get_subfields(*codes):
            yield field.tag, path, value.strip()


counts = Counter()
for store, table in TABLES.items():
    for batch in AdapterStore(table, store).stream_active_namespace_records():
        for content in batch.column("content").to_pylist():
            if content:
                for tag, path, text in period_strings(parse_single_marc_record(content)):
                    counts[(store, tag, path, text)] += 1
    print(f"{store}: {sum(n for (s, *_), n in counts.items() if s == store)} strings")

with OUT.open("w", encoding="utf-8") as f:
    for (store, tag, path, text), n in counts.items():
        source = "axiell" if store == "axiell" else "marc"
        f.write(json.dumps({"store": store, "tag": tag, "path": path, "source": source, "text": text, "n": n}, ensure_ascii=False) + "\n")
print(f"{sum(counts.values())} strings, {len(counts)} rows -> {OUT}")

ebsco: 236592 strings
axiell: 208829 strings
folio: 1102450 strings
1547871 strings, 80634 rows -> data/periods.jsonl
